# Data Collection
LSE_ID =  250094127

This notebook collects raw movie data from The Movie Database (TMDB) API. I gather the top revenue-generating movies released between 2015 and 2025, save the raw discovery results to JSON files by year, then fetch full details for each movie and save those to a single combined file for use in later analysis notebooks.

## Connecting to the API

First, I **import** the relevant libraries and make a connection to the API I'll use

In [12]:
import requests 
import os 
from dotenv import load_dotenv
import json

Next, I load my API credentials from a `.env` file (rather than hardcoding them) using `load_dotenv`, then store my access token and set up the headers I'll pass with every request to the TMDB API.

In [13]:
load_dotenv()

True

In [14]:
API_ACCESS_TOKEN = os.environ["API_ACCESS_TOKEN"]

In [15]:
import requests

BASE_URL = "https://api.themoviedb.org/3"
HEADERS = {
    "Authorization": f"Bearer {API_ACCESS_TOKEN}",
    "accept": "application/json"
}


I'll be examining **10 years** worth of movie data. 

In [16]:
years = []
for i in range(2015, 2025+1):
    years.append(i)

years

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

## Making the json files 

I run a for loop to get the first 100 results of the movies with the most **revenue** during the years 2015 until 2020. First, I run the years list. For each year, if it is the first page, I create a list called results from the response value in the results key. If its not the first page, then I'll just extend the list that was made from the first page, which means that each item in the list results will be added separately. 

I request **5 pages** per year because TMDB's `discover` endpoint returns 20 results per page, so 5 pages gets me up to 100 movies — enough to comfortably capture each year's top earners without pulling in films far down the revenue ranking. Since I sort by `revenue.desc`, page 1 already holds the highest-grossing films for that year; extending across pages just continues down the same ranked list rather than mixing in a different ordering.

In [17]:
url_discover = f"{BASE_URL}/discover/movie"
response_years = [] 
for year in years: 
    for page in range(1,6):
        parameters_discover = {
                "primary_release_date.gte": f"{year}-01-01",
                "primary_release_date.lte": f"{year}-12-31",
                "sort_by": "revenue.desc",
                "language": "en-US",
                "page": page
            }
        response_discover = requests.get(url_discover, headers=HEADERS, params=parameters_discover)
        if page == 1: 
            results = response_discover.json()["results"]
        else:
            results.extend(response_discover.json()["results"])
    
    response_years.append(results)


In [18]:
response_years[0]

[{'adult': False,
  'backdrop_path': '/k6EOrckWFuz7I4z4wiRwz8zsj4H.jpg',
  'genre_ids': [12, 28, 878],
  'id': 140607,
  'title': 'Star Wars: The Force Awakens',
  'original_language': 'en',
  'original_title': 'Star Wars: The Force Awakens',
  'overview': 'Thirty years after defeating the Galactic Empire, Han Solo and his allies face a new threat from the evil Kylo Ren and his army of Stormtroopers.',
  'popularity': 20.7796,
  'poster_path': '/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg',
  'release_date': '2015-12-15',
  'softcore': False,
  'video': False,
  'vote_average': 7.248,
  'vote_count': 20693},
 {'adult': False,
  'backdrop_path': '/dF6FjTZzRTENfB4R17HDN20jLT2.jpg',
  'genre_ids': [12, 878, 53],
  'id': 135397,
  'title': 'Jurassic World',
  'original_language': 'en',
  'original_title': 'Jurassic World',
  'overview': 'Twenty-two years after the events of Jurassic Park, Isla Nublar now features a fully functioning dinosaur theme park, Jurassic World, as originally envisioned by John

After, I run another for loop. This time, I use the **zip** function, that runs over the years and their responses simultaneously, which lets me create a json file for each year.

In [19]:
for year, response_year in zip(years,response_years): 
    with open(f"../data/raw/{year}.json", "w") as f:
        json.dump(response_year, f)

The JSON files are now saved, one per year. As a sanity check, I look at a single movie entry to see what fields are available at this stage — this initial `discover` response is missing some fields I need for analysis (e.g. budget, runtime), which is why I fetch full movie details next.

In [20]:
response_years[0][0]

{'adult': False,
 'backdrop_path': '/k6EOrckWFuz7I4z4wiRwz8zsj4H.jpg',
 'genre_ids': [12, 28, 878],
 'id': 140607,
 'title': 'Star Wars: The Force Awakens',
 'original_language': 'en',
 'original_title': 'Star Wars: The Force Awakens',
 'overview': 'Thirty years after defeating the Galactic Empire, Han Solo and his allies face a new threat from the evil Kylo Ren and his army of Stormtroopers.',
 'popularity': 20.7796,
 'poster_path': '/wqnLdwVXoBjKibFRR5U3y0aDUhs.jpg',
 'release_date': '2015-12-15',
 'softcore': False,
 'video': False,
 'vote_average': 7.248,
 'vote_count': 20693}

This last loop is used so that I get further details about each movie collected. First, I create a dictionary that will only hold a list of all the movies. This nested loop runs over each year, and fetches the id of every movie in that year. This id is then used so as to use it the **movie** endpoint. After that, I create a json file only containing movie details over all those years. 

Since each year contributes up to 100 movies across 11 years, this loop makes on the order of **1,100 individual API calls** — one `movie/{id}` request per film — which is why it's kept as a separate step rather than folded into the discovery loop above.

In [21]:
results_movies = {"movies": []}
for response_year in response_years:
    for movie in response_year:
        url_movies = f"{BASE_URL}/movie/{movie["id"]}"
        parameters_movies= {"language": "en-US"}
        response_movies= requests.get(url_movies, headers=HEADERS, params=parameters_movies)
        results_movies["movies"].append(response_movies.json())

In [22]:
with open(f"../data/raw/movie_details.json", "w")as f:
        json.dump(results_movies, f) 
        

Finally, I check how many movies were collected in total across all 11 years (up to 100 movies per year, from 2015 through 2025).

In [25]:
len(results_movies["movies"])

1100